# MFCC-10 CNN UUV Detection

Trains temporal Conv1D models on MFCC-10 features for multilabel and binary UUV detection across normal, M-filtered, and W-filtered splits.

In [ ]:
from pathlib import Path
import os
import sys
from urllib.request import urlretrieve

UTILS_GITHUB_RAW_BASE_URL = ""  # Set to the repository raw-file URL when utility files are not synced to Colab.


def find_or_fetch_utility(relative_path: str) -> Path:
    roots = [Path.cwd(), Path.cwd().parent, Path("/content"), Path("/content/drive/MyDrive/STUDA/src")]
    for root in roots:
        candidate = root / relative_path
        if candidate.is_file():
            return candidate
    if UTILS_GITHUB_RAW_BASE_URL:
        destination = Path("/content") / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f"{UTILS_GITHUB_RAW_BASE_URL.rstrip('/')}/{relative_path}", destination)
        return destination
    raise FileNotFoundError(
        f"Missing {relative_path}. Sync it with the VS Code Colab extension, upload it to /content, "
        "mount Drive at /content/drive/MyDrive/STUDA/src, or set UTILS_GITHUB_RAW_BASE_URL."
    )


common_utils_path = find_or_fetch_utility("utils/common_utils.py")
mfcc_cnn_utils_path = find_or_fetch_utility("CNN/mfcc_cnn_utils.py")
for utility_dir in (common_utils_path.parent, mfcc_cnn_utils_path.parent):
    if str(utility_dir) not in sys.path:
        sys.path.insert(0, str(utility_dir))

from common_utils import (
    configure_kaggle_access, evaluate_models_for_variants, extract_zip, plot_training_histories,
    prepare_mfcc_dataset_variants, save_keras_artifacts, train_keras_models_for_variants, zip_artifacts,
)
from mfcc_cnn_utils import build_mfcc_cnn_models_for_variants, get_mfcc_cnn_callbacks


In [ ]:
from google.colab import files
from IPython.display import display

DATASET_KEY = "mfcc10"
DATASET_LABEL = "MFCC-10"
DATASET_SLUG = "pawedyrda/mfcc10"
ARCHIVE_PATH = Path("/content/mfcc10.zip")
EPOCHS = 50
BATCH_SIZE = 64

configure_kaggle_access(secret_name="Kaggle")


In [ ]:
!kaggle datasets download -d {DATASET_SLUG} -p /content --force
DATA_PATH = extract_zip(ARCHIVE_PATH, "/content")
print(f"Dataset extracted to: {DATA_PATH}")


In [ ]:
variants = prepare_mfcc_dataset_variants(DATA_PATH)
print(f"MFCC input shape: {variants.normal.train_data.shape[1:]}")


In [ ]:
multilabel_models = build_mfcc_cnn_models_for_variants(variants, model_type="multilabel")
multilabel_histories = train_keras_models_for_variants(
    multilabel_models, variants, model_type="multilabel", epochs=EPOCHS,
    batch_size=BATCH_SIZE, callback_factory=get_mfcc_cnn_callbacks,
)


In [ ]:
binary_models = build_mfcc_cnn_models_for_variants(variants, model_type="binary")
binary_histories = train_keras_models_for_variants(
    binary_models, variants, model_type="binary", epochs=EPOCHS,
    batch_size=BATCH_SIZE, callback_factory=get_mfcc_cnn_callbacks,
)


In [ ]:
multilabel_results = evaluate_models_for_variants(multilabel_models, variants, "multilabel", DATASET_LABEL)
binary_results = evaluate_models_for_variants(binary_models, variants, "binary", DATASET_LABEL)
display(multilabel_results)
display(binary_results)

plot_training_histories(multilabel_histories, f"Multilabel MFCC-CNN Training Curves - {DATASET_LABEL}")
plot_training_histories(binary_histories, f"Binary MFCC-CNN Training Curves - {DATASET_LABEL}")

save_dir = save_keras_artifacts(
    f"/content/saved_artifacts/cnn_{DATASET_KEY}", DATASET_KEY, multilabel_models, binary_models,
    multilabel_histories, binary_histories, multilabel_results, binary_results,
)
archive_path = zip_artifacts(save_dir, f"/content/cnn_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))
